# Hong Kong residential data: see the whole dataset

## Goal
Get a compact group overview, a clean numeric DataFrame, and a searchable indicator catalogue. Then select a group, suffixes and dates. Data comes from the existing GitHub publication; this notebook does not run collectors or create new economic estimates.

The saved outputs are a run-time snapshot. Running the cells again retrieves the current published snapshot. [Grouped indicator guide](https://github.com/Hogan-cyber/Real-Estate-Data#指标组速查) · [Downloadable dataset](https://github.com/Hogan-cyber/Real-Estate-Data/releases/tag/residential-data-latest)

## Setup
Use Python 3.10+. Install once below; after upgrading in an existing session, restart the kernel. On subsequent visits, skip the installation cell.

In [1]:
import sys
assert sys.version_info >= (3, 10), "Select a Python 3.10+ kernel."
%pip install --quiet --upgrade "https://github.com/Hogan-cyber/Real-Estate-Data/releases/download/residential-data-latest/hk_property_data-0.5.0-py3-none-any.whl"


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: /tmp/hk-readme-quickstart-20261109/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Steps
### 1. Read the full dataset
`all_data` has a date index and numeric indicator columns. `catalog` explains the IDs, units, frequencies and sources. Different frequencies align to different dates, creating `NaN` cells; no filling or resampling occurs. The first read uses the network; later calls share the same session snapshot.

In [2]:
import pandas as pd
import hk_property_data as hk

try:
    from IPython.display import display
except ImportError:
    display = print

market = hk.Residential()
catalog = market.catalog()
all_data = market.history(labels="id")

print("Package:", hk.__version__)
print("Snapshot generated:", all_data.attrs["data_generated_at"])
print(f"{len(catalog)} indicators; {len(all_data):,} date rows")

Package: 0.5.0
Snapshot generated: 2026-11-08T16:33:31.994076+00:00
163 indicators; 6,216 date rows


### 2. See the whole picture by indicator group
The small prefix map is a navigation aid. Related area classes and districts stay together; existing mortgage estimates and their three input shares have their own group. Future unfamiliar IDs remain visible as `Other / new indicators`.

`D` = daily; `M` = monthly; `Q` = quarterly; `A` = annual; `E` = effective-date events. Records include original missing observations; available values exclude them. Group date ranges cover the union of recorded dates, not guaranteed non-missing coverage for every member.

In [3]:
GROUP_PREFIXES = {
    "Transactions": ("hk_residential_sales_", "hk_residential_primary_sales_", "hk_residential_secondary_sales_", "hk_all_building_sales_"),
    "Average rents": ("rvd_1_1_",),
    "Average prices": ("rvd_1_2_",),
    "Rent indices": ("rvd_1_3_",),
    "Price indices": ("rvd_1_4_",),
    "Popular developments": ("rvd_1_5_",),
    "Annual supply": ("rvd_stock_", "rvd_completions_", "rvd_vacancy_", "rvd_takeup_"),
    "Bank mortgages": ("hk_mortgage_",),
    "Mortgage registrations": ("hk_all_property_",),
    "Negative equity": ("hk_negative_equity_",),
    "Money supply": ("hk_m1_", "hk_m2_", "hk_m3_"),
    "HIBOR": ("hk_hibor_",),
    "Prime rate": ("hk_prime_rate_hsbc",),
}
MODEL_INPUT_SHARES = {
    "hk_residential_sales_share",
    "hk_bank_mortgage_refinancing_count_share",
    "hk_bank_mortgage_refinancing_amount_share",
}


def indicator_group(metric_id):
    if metric_id.endswith("_estimate") or metric_id in MODEL_INPUT_SHARES:
        return "Mortgage estimates"
    for group, prefixes in GROUP_PREFIXES.items():
        if metric_id.startswith(prefixes):
            return group
    return "Other / new indicators"  # Future indicators remain visible.


catalog["group"] = catalog.index.map(indicator_group)
overview = catalog.groupby(["group", "frequency"], sort=True).agg(
    indicators=("metric_name", "size"),
    records=("observations", "sum"),
    available_values=("non_missing", "sum"),
    first_date=("first_date", "min"),
    last_date=("last_date", "max"),
)
display(overview)

indicators  records  available_values  \
group                  frequency                                          
Annual supply          A                  27      654               654   
Average prices         M                  15     4455              4426   
Average rents          M                  15     4455              4448   
Bank mortgages         M                  33     5505              5387   
HIBOR                  M                   2      596               596   
                       D                   7    42831             37756   
Money supply           M                   9     2673              2673   
Mortgage estimates     M                  10     2980              1540   
Mortgage registrations M                   4     1192              1192   
Negative equity        Q                   7      243               212   
Popular developments   M                   9     2673              2673   
Price indices          M                   8     2376              2376   
Prime rate             M                   1      298               298   
                       E                   1       33                33   
Rent indices           M                   8     2376              2376   
Transactions           M                   7     2082              2082   

                                 first_date  last_date  
group                  frequency                        
Annual supply          A         2002-12-31 2026-09-30  
Average prices         M         2002-01-31 2026-09-30  
Average rents          M         2002-01-31 2026-09-30  
Bank mortgages         M         2002-01-31 2026-09-30  
HIBOR                  M         2002-01-31 2026-10-31  
                       D         2002-01-02 2026-11-06  
Money supply           M         2002-01-31 2026-09-30  
Mortgage estimates     M         2002-01-31 2026-10-31  
Mortgage registrations M         2002-01-31 2026-10-31  
Negative equity        Q         2016-12-31 2026-09-30  
Popular developments   M         2002-01-31 2026-09-30  
Price indices          M         2002-01-31 2026-09-30  
Prime rate             M         2002-01-31 2026-10-31  
                       E         2002-11-08 2025-10-31  
Rent indices           M         2002-01-31 2026-09-30  
Transactions           M         2002-01-31 2026-10-31

### 3. Preview six monthly indicators
First inspect their units and coverage, then read the latest six date rows. This is a selection of existing series, not a composite index. Blank cells are preserved, and the final date need not have a value for every series.

In [4]:
core_ids = [
    "rvd_1_4_all_classes",
    "rvd_1_3_all_classes",
    "hk_residential_sales_count",
    "hk_mortgage_drawn_amount",
    "hk_hibor_1m",
    "hk_m3_hkd",
]
catalog_columns = ["metric_name", "unit", "frequency", "first_date", "last_date"]
display(catalog.loc[core_ids, catalog_columns])
core = market.history(series=core_ids, labels="name")
display(core.tail(6))

,metric_name,unit,frequency,first_date,last_date
metric_id,,,,,
rvd_1_4_all_classes,Price index — All Classes,1999=100,M,2002-01-31,2026-09-30
rvd_1_3_all_classes,Rental index — All Classes,1999=100,M,2002-01-31,2026-09-30
hk_residential_sales_count,住宅楼宇买卖合约登记宗数,份,M,2002-01-31,2026-10-31
hk_mortgage_drawn_amount,New mortgage loans drawn down: amount,HKD million,M,2002-01-31,2026-09-30
hk_hibor_1m,1-month HIBOR (monthly average),% p.a.,M,2002-01-31,2026-10-31
hk_m3_hkd,Hong Kong dollar M3,HKD million,M,2002-01-31,2026-09-30


,Price index — All Classes,Rental index — All Classes,住宅楼宇买卖合约登记宗数,New mortgage loans drawn down: amount,1-month HIBOR (monthly average),Hong Kong dollar M3
Date,,,,,,
2026-05-31,322.6,203.9,7138.0,23573.0,2.597065,9224978.799
2026-06-30,323.4,205.8,7650.0,27720.0,2.800825,9295676.592
2026-07-31,320.1,207.4,4462.0,29371.0,2.725410,9386805.378
2026-08-31,320.2,211.2,4019.0,32768.0,2.667363,9586156.603
2026-09-30,320.8,212.2,4761.0,25291.0,2.904789,9505133.000
2026-10-31,NaN,NaN,4512.0,NaN,2.768845,NaN


### 4. Choose a group and dates
Copy a group name from `overview`. Set `end = None` to include all available subsequent dates. `df` is a normal pandas DataFrame. For mixed-frequency groups such as HIBOR, add `& catalog["frequency"].eq("M")` to the member selection if you want monthly data only.

In [5]:
group_name = "Price indices"
start, end = "2025-01", None  # None includes all subsequently available dates.
members = catalog.loc[catalog["group"].eq(group_name)]
if members.empty:
    raise ValueError(f"Unknown or empty group: {group_name!r}; inspect overview")
display(members[catalog_columns])
df = market.history(series=members.index.tolist(), start=start, end=end, labels="id")
display(df.tail(6))

,metric_name,unit,frequency,first_date,last_date
metric_id,,,,,
rvd_1_4_all_classes,Price index — All Classes,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_class_a,Price index — Class A,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_class_b,Price index — Class B,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_class_c,Price index — Class C,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_class_d,Price index — Class D,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_class_e,Price index — Class E,1999=100,M,2002-01-31,2026-09-30
rvd_1_4_classes_a_b_c,"Price index — Classes A, B & C",1999=100,M,2002-01-31,2026-09-30
rvd_1_4_classes_d_e,Price index — Classes D & E,1999=100,M,2002-01-31,2026-09-30


,rvd_1_4_all_classes,rvd_1_4_class_a,rvd_1_4_class_b,rvd_1_4_class_c,rvd_1_4_class_d,rvd_1_4_class_e,rvd_1_4_classes_a_b_c,rvd_1_4_classes_d_e
Date,,,,,,,,
2026-04-30,317.9,336.8,311.8,302.5,282.8,277.2,320.1,281.4
2026-05-31,322.6,342.6,315.8,306.9,286.8,281.6,324.9,285.5
2026-06-30,323.4,343.3,316.0,309.3,290.0,283.7,325.5,288.5
2026-07-31,320.1,340.4,312.0,305.7,288.4,283.6,322.1,287.3
2026-08-31,320.2,340.4,312.0,305.7,290.4,285.4,322.1,289.2
2026-09-30,320.8,340.4,312.6,307.1,293.0,286.8,322.6,291.5


### 5. Pick suffixes or a list of indicators
`rvd_1_2_` selects average prices; `_kowloon` selects Kowloon, retaining its area classes. Only IDs found in the catalogue are sent to the reader. Date bounds are inclusive; `2025-12` includes the whole month. Use `labels="name"` for readable names, or `labels="id"` for stable selection keys.

In [6]:
# Example: classes A-E in Kowloon, average prices only.
area_ids = catalog.index[
    catalog.index.str.startswith("rvd_1_2_")
    & catalog.index.str.endswith("_kowloon")
].tolist()
area_prices = market.history(series=area_ids, start="2025-01", labels="id")
display(area_prices.tail(3))

selected_ids = ["rvd_1_4_all_classes", "rvd_1_3_all_classes"]
selected = market.history(series=selected_ids, start="2025-01", end="2025-12", labels="id")
display(selected.tail(3))

,rvd_1_2_class_a_kowloon,rvd_1_2_class_b_kowloon,rvd_1_2_class_c_kowloon,rvd_1_2_class_d_kowloon,rvd_1_2_class_e_kowloon
Date,,,,,
2026-07-31,128528.0,134206.0,173155.0,194650.0,322918.0
2026-08-31,129742.0,136408.0,180697.0,213946.0,257132.0
2026-09-30,134337.0,136529.0,172982.0,188500.0,263433.0


,rvd_1_4_all_classes,rvd_1_3_all_classes
Date,,
2025-10-31,294.9,199.9
2025-11-30,298.7,200.4
2025-12-31,299.6,200.5


### 6. Search and inspect a source record
Search by a literal word in the ID or name. `observations()` provides the source URL, status and original metadata for a particular period. HKMA bank lending and the existing formula-based money-lender residuals have different meanings; estimates are not official money-lender observations.

In [7]:
keyword = "refinancing"
matches = catalog.loc[
    catalog.index.str.contains(keyword, case=False, regex=False)
    | catalog["metric_name"].str.contains(keyword, case=False, na=False, regex=False)
]
display(matches[["group"] + catalog_columns])
source = market.observations(series="hk_mortgage_drawn_amount", start="2025-01", end="2025-01")
display(source[["date", "metric_name", "value", "unit", "status", "source_url"]])

,group,metric_name,unit,frequency,first_date,last_date
metric_id,,,,,,
hk_bank_mortgage_purchase_drawn_amount_estimate,Mortgage estimates,Bank excluding refinancing: amount estimate,HKD million,M,2002-01-31,2026-10-31
hk_bank_mortgage_purchase_drawn_count_estimate,Mortgage estimates,Bank excluding refinancing: count estimate,estimated loans,M,2002-01-31,2026-10-31
hk_bank_mortgage_refinancing_amount_share,Mortgage estimates,Bank refinancing share of approvals: amount,fraction,M,2002-01-31,2026-10-31
hk_bank_mortgage_refinancing_count_share,Mortgage estimates,Bank refinancing share of approvals: count,fraction,M,2002-01-31,2026-10-31
hk_mortgage_approved_refinancing_amount,Bank mortgages,Refinancing mortgage approvals: amount,HKD million,M,2016-12-31,2026-09-30
hk_mortgage_approved_refinancing_count,Bank mortgages,Refinancing mortgage approvals: count,loans,M,2016-12-31,2026-09-30


,date,metric_name,value,unit,status,source_url
0,2025-01-31,New mortgage loans drawn down: amount,15590.0,HKD million,Published,https://www.hkma.gov.hk/media/eng/doc/market-d...


## Checks
These checks confirm the table shape and that grouping retains every indicator. They do not independently reconcile each value with its official source.

In [8]:
assert isinstance(all_data, pd.DataFrame)
assert isinstance(all_data.index, pd.DatetimeIndex)
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in all_data.dtypes)
assert overview["indicators"].sum() == len(catalog) == all_data.shape[1]
assert overview["records"].sum() == catalog["observations"].sum()
assert not catalog.index.duplicated().any()
print("Numeric DataFrame and complete indicator-group coverage checked.")

Numeric DataFrame and complete indicator-group coverage checked.


## Next Steps
Open `overview`, `catalog`, `all_data`, `core` or `df` in Jupyter / Spyder. The full data remains in the variables even if a display abbreviates rows or columns.

To refresh explicitly, run the following and then rerun the grouping and selection cells. Previous DataFrames do not change automatically.
```python
all_data = market.history(labels="id", refresh=True)
catalog = market.catalog()
```
The shared Arrow cache is bounded at 8 MiB; returned DataFrames use additional memory. The reader does not save dataset files to disk. [Python guide](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/PYTHON_PACKAGE.md) · [Full source-linked dictionary](https://github.com/Hogan-cyber/Real-Estate-Data/blob/residential-data/INDICATORS.md)
